# D7 — full-scale structural runtime and peak-RSS audit
**This is a NEW measured rerun, not the historical runtime.** The data must already exist in Colab. Outputs are aggregate-only. Do not claim all of D7 is closed until the full-scale run finishes and a versioned code archive exists.

Run the same original source file used in the study, ideally copied to local `/content` before profiling; Google Drive I/O changes performance.

In [1]:
%pip -q install duckdb psutil pandas numpy

In [6]:
"""D7: full-flow wall-clock and sampled peak-RSS profiler for the corrected
source-degree structural pipeline. Does not export raw IPs or per-IP labels.

Dependencies: pip install duckdb pandas numpy psutil
Expected data: NF-UQ-NIDS-v2.csv with Dataset, IPV4_SRC_ADDR, IPV4_DST_ADDR, Label.
Run in Colab with full RAW CSV at /content/nf_uq_nids/NF-UQ-NIDS-v2.csv.
Do not conflate these new benchmark numbers with historical experiment runtimes.
"""
import csv
import hashlib
import json
import os
import platform
import sys
import threading
import time
import tempfile
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

COHORTS = ('NF-ToN-IoT-v2','NF-CSE-CIC-IDS2018-v2')
POLICY = dict(delta_tc=.05,gamma_min=3.0,k_min=2,top_k=30,mgr_low=2.0,mgr_strong=5.0)


def structural_diagnosis(deg):
    """Degree-only rule; NO labels in signature or branch conditions."""
    deg = np.asarray(deg,dtype=np.float64)
    if deg.size==0 or np.any(deg<=0):raise ValueError('Only active sources with positive degree')
    q90,q95,q97 = np.percentile(deg,[90,95,97],method='linear')
    tc = bool(abs(q97-q90)/max(q97,1.0) < POLICY['delta_tc'])
    top = np.sort(deg[deg>1.0])[::-1][:POLICY['top_k']]
    ratios = top[:-1]/top[1:] if len(top)>1 else np.array([])
    mgr = float(ratios.max()) if len(ratios) else None
    eligible = [(i+1,float(r),float(top[i])) for i,r in enumerate(ratios)
                if i+1 >= POLICY['k_min'] and r>POLICY['gamma_min']]
    if not tc:
        threshold=float(q95); choice='P95_STABLE'
    elif mgr is not None and mgr>=POLICY['mgr_strong'] and eligible:
        rank,ratio,threshold = min(eligible,key=lambda e:e[0]);choice='GAP_RECOVERY_CANDIDATE'
    else:
        threshold=None;choice='ABSTAIN'
    out = dict(n_sources=int(len(deg)),P90=float(q90),P95=float(q95),P97=float(q97),
               TC=tc,MGR=mgr,n_eligible_gap_ranks=len(eligible),decision=choice,
               threshold=threshold,selected=int(np.count_nonzero(deg>=threshold)) if threshold is not None else None)
    return out


class StageProfiler:
    def __init__(self): self.rows=[];self.proc=psutil.Process(os.getpid())

    @contextmanager
    def stage(self,name):
        stop=threading.Event(); readings=[]
        def observe():
            while not stop.is_set():
                try:readings.append(self.proc.memory_info().rss)
                except (psutil.NoSuchProcess,psutil.AccessDenied):pass
                stop.wait(.10)
        monitor=threading.Thread(target=observe,daemon=True)
        start=time.perf_counter();monitor.start()
        try:yield
        finally:
            stop.set();monitor.join(timeout=1)
            try:readings.append(self.proc.memory_info().rss)
            except (psutil.NoSuchProcess,psutil.AccessDenied):pass
            self.rows.append(dict(stage=name,wall_seconds=round(time.perf_counter()-start,3),
                                  peak_process_rss_gib=round(max(readings,default=0)/2**30,3),
                                  n_rss_samples=len(readings)))


def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        for chunk in iter(lambda:fh.read(8<<20),b''):h.update(chunk)
    return h.hexdigest()


def run(input_path='/content/nf_uq_nids/NF-UQ-NIDS-v2.csv',
        out_dir='/content/drive/MyDrive/paper2_gsd_results/d7_profile',
        memory_limit='10GB', threads=2, hash_input=False):
    # Program must be executed on the ORIGINAL file to provide full-scale timings.
    try:import duckdb
    except ImportError as exc: raise RuntimeError('Run %pip install duckdb psutil first') from exc
    path=Path(input_path)
    if not path.is_file():raise FileNotFoundError(f'Full raw input not found: {path}')
    od=Path(out_dir);od.mkdir(parents=True,exist_ok=True)
    # Per-IP scratch tables are strictly local and deleted before the run returns.
    local_scratch=Path('/content') if Path('/content').is_dir() else Path('/tmp')
    scratch=tempfile.TemporaryDirectory(prefix='d7_private_',dir=str(local_scratch))
    private=Path(scratch.name)
    (private/'spill').mkdir()
    pr=StageProfiler();conn=duckdb.connect(str(private/'temporary.duckdb'))
    if not (memory_limit[:-2].isdigit() and memory_limit.endswith('GB')):
        raise ValueError('memory_limit must be integer GB, e.g. 10GB')
    if not (1 <= int(threads) <= 64):raise ValueError('threads must be 1..64')
    conn.execute(f"SET memory_limit='{memory_limit}'")
    conn.execute(f"SET threads={int(threads)}")
    tmp_path=str(private/'spill').replace("'","''")
    conn.execute(f"SET temp_directory='{tmp_path}'")
    scan="read_csv_auto(?, header=true, all_varchar=true)"
    with pr.stage('01_scan_and_cohort_row_counts'):
        nrows=conn.execute(f'''SELECT Dataset AS cohort, COUNT(*) AS n
                   FROM {scan} WHERE Dataset IN (?,?) GROUP BY Dataset''',
                   [str(path),*COHORTS]).fetchall()
    with pr.stage('02_distinct_source_destination_edges'):
        conn.execute('DROP TABLE IF EXISTS edge_pairs')
        conn.execute(f'''CREATE TABLE edge_pairs AS SELECT Dataset AS cohort,
                   IPV4_SRC_ADDR AS src, IPV4_DST_ADDR AS dst
                   FROM {scan}
                   WHERE Dataset IN (?,?) AND IPV4_SRC_ADDR IS NOT NULL
                     AND IPV4_DST_ADDR IS NOT NULL
                   GROUP BY Dataset, IPV4_SRC_ADDR, IPV4_DST_ADDR''',
                   [str(path),*COHORTS])
    with pr.stage('03_source_outdegree_aggregation'):
        conn.execute('DROP TABLE IF EXISTS degrees')
        conn.execute('''CREATE TABLE degrees AS SELECT cohort,src,COUNT(*)::BIGINT AS out_degree
                        FROM edge_pairs GROUP BY cohort,src''')
    with pr.stage('04_degree_only_diagnostic'):
        results={}
        for cohort in COHORTS:
            vals=conn.execute('SELECT out_degree FROM degrees WHERE cohort=?',[cohort]).fetchnumpy()['out_degree']
            results[cohort]=structural_diagnosis(vals)
            results[cohort]['n_edges']=int(conn.execute('SELECT COUNT(*) FROM edge_pairs WHERE cohort=?',[cohort]).fetchone()[0])
    with pr.stage('05_posthoc_source_labels_separate_from_decision'):
        conn.execute('DROP TABLE IF EXISTS source_labels')
        conn.execute(f'''CREATE TABLE source_labels AS SELECT Dataset AS cohort,
            IPV4_SRC_ADDR AS src,
            MAX(CASE WHEN TRY_CAST(Label AS DOUBLE)=1.0 THEN 1 ELSE 0 END) AS attacker
            FROM {scan} WHERE Dataset IN (?,?) AND IPV4_SRC_ADDR IS NOT NULL
            GROUP BY Dataset, IPV4_SRC_ADDR''',[str(path),*COHORTS])
        for cohort in COHORTS:
            tau=results[cohort]['threshold']
            results[cohort]['attack_sources']=int(conn.execute('SELECT SUM(attacker) FROM source_labels WHERE cohort=?',[cohort]).fetchone()[0] or 0)
            if tau is not None:
                tp=conn.execute('''SELECT SUM(l.attacker) FROM degrees d
                   LEFT JOIN source_labels l ON d.cohort=l.cohort AND d.src=l.src
                   WHERE d.cohort=? AND d.out_degree>=?''',[cohort,tau]).fetchone()[0]
                results[cohort]['selected_attack_sources']=int(tp or 0)
    env=dict(python=sys.version.split()[0],os=platform.platform(),
             cpu=platform.processor(),logical_cores=psutil.cpu_count(),
             total_ram_gib=round(psutil.virtual_memory().total/2**30,2),
             pandas=pd.__version__,numpy=np.__version__,duckdb=duckdb.__version__,
             full_input_basename=path.name,full_input_bytes=path.stat().st_size,
             policy=POLICY,memory_limit=memory_limit,threads=threads,
             scope='NEW benchmark; NOT historical wall clock or memory')
    if hash_input:
        with pr.stage('06_input_sha256_optional'):
            env['full_input_sha256']=hash_file(path)
    else:env['full_input_sha256']='NOT_COMPUTED; use hash_input=True for archival integrity'
    (od/'d7_profile_environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')
    (od/'d7_profile_aggregate.json').write_text(json.dumps(dict(cohort_rows=dict(nrows),results=results),indent=2),encoding='utf-8')
    with (od/'d7_profile_stage_runtime.csv').open('w',newline='',encoding='utf-8') as fh:
        w=csv.DictWriter(fh,fieldnames=['stage','wall_seconds','peak_process_rss_gib','n_rss_samples']);w.writeheader();w.writerows(pr.rows)
    conn.close()
    scratch.cleanup()
    print('Produced aggregate-only outputs in:',od)
    print('Note: process-RSS memory, not GPU RAM or total DuckDB spill usage.')
    for key,value in results.items():print(key,value)
    return results,pr.rows

if False and __name__=='__main__':
    import argparse
    p=argparse.ArgumentParser()
    p.add_argument('--input',default='/content/nf_uq_nids/NF-UQ-NIDS-v2.csv')
    p.add_argument('--out',default='/content/d7_profile')
    p.add_argument('--memory-limit',default='10GB')
    p.add_argument('--threads',type=int,default=2)
    p.add_argument('--hash-input',action='store_true')
    a=p.parse_args()
    run(a.input,a.out,a.memory_limit,a.threads,a.hash_input)


In [4]:
%pip -q install duckdb psutil pandas numpy

In [9]:

%pip -q install kaggle

from pathlib import Path
from google.colab import files
import subprocess

# 1. Check whether the dataset already exists.
locations = [
    Path("/content/nf_uq_nids/NF-UQ-NIDS-v2.csv"),
    Path("/content/drive/MyDrive/nf_uq_nids/NF-UQ-NIDS-v2.csv"),
]

existing = next((p for p in locations if p.is_file()), None)

if existing is None:
    # 2. Configure Kaggle authentication.
    kaggle_dir = Path("/root/.kaggle")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    credential = kaggle_dir / "kaggle.json"

    if not credential.exists():
        alt = Path("/root/.config/kaggle/kaggle.json")
        if alt.exists():
            credential.write_bytes(alt.read_bytes())
        else:
            print("Upload kaggle.json")
            uploaded = files.upload()
            credential.write_bytes(uploaded["kaggle.json"])

    credential.chmod(0o600)

    # 3. Download directly into the current Colab runtime.
    folder = Path("/content/nf_uq_nids")
    folder.mkdir(parents=True, exist_ok=True)

    subprocess.run([
        "kaggle", "datasets", "download",
        "-d",
        "aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset",
        "-p", str(folder),
        "--unzip"
    ], check=True)

    matches = list(folder.rglob("NF-UQ-NIDS-v2.csv"))
    if not matches:
        raise FileNotFoundError(
            "Dataset downloaded but expected CSV was not found."
        )
    existing = matches[0]

INPUT = str(existing)

print("DATASET FOUND:", INPUT)
print("SIZE (GB):", round(existing.stat().st_size / 1024**3, 2))
print("run() loaded:", callable(globals().get("run")))


Upload kaggle.json


Saving kaggle.json to kaggle.json
DATASET FOUND: /content/nf_uq_nids/NF-UQ-NIDS-v2.csv
SIZE (GB): 12.79
run() loaded: True


In [10]:

print("run() available:", callable(globals().get("run")))
print("INPUT:", INPUT)


run() available: True
INPUT: /content/nf_uq_nids/NF-UQ-NIDS-v2.csv


In [11]:

from pathlib import Path

assert Path(INPUT).is_file(), f"Dataset missing: {INPUT}"

results, runtime_rows = run(
    INPUT,
    "/content/drive/MyDrive/paper2_gsd_results/d7_profile",
    memory_limit="10GB",
    threads=2,
    hash_input=False
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Produced aggregate-only outputs in: /content/drive/MyDrive/paper2_gsd_results/d7_profile
Note: process-RSS memory, not GPU RAM or total DuckDB spill usage.
NF-ToN-IoT-v2 {'n_sources': 23079, 'P90': 1.0, 'P95': 1.0, 'P97': 1.0, 'TC': True, 'MGR': 12.466183574879228, 'n_eligible_gap_ranks': 1, 'decision': 'GAP_RECOVERY_CANDIDATE', 'threshold': 214.0, 'selected': 11, 'n_edges': 31645, 'attack_sources': 31, 'selected_attack_sources': 10}
NF-CSE-CIC-IDS2018-v2 {'n_sources': 224568, 'P90': 10.0, 'P95': 22.0, 'P97': 36.0, 'TC': False, 'MGR': 1.1860028182245186, 'n_eligible_gap_ranks': 0, 'decision': 'P95_STABLE', 'threshold': 22.0, 'selected': 11571, 'n_edges': 1872361, 'attack_sources': 648, 'selected_attack_sources': 301}
